In [1]:
# FinGuard - Model Interpretability with SHAP
# Explains WHY the model predicts fraud

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import shap
import os
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
os.makedirs('./reports/figures', exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


g:\Projects Ai-ML\Fraud Detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 1. LOAD DATA & BEST MODEL
# ============================================================
print("=" * 60)
print("FinGuard - Model Interpretability (SHAP)")
print("=" * 60)

print("\n[1] Loading data and best model...")

X_test = pd.read_csv('./data/processed/X_test.csv')
X_test_scaled = pd.read_csv('./data/processed/X_test_scaled.csv')
y_test = pd.read_csv('./data/processed/y_test.csv').squeeze()

with open('./models_saved/model_metadata.json') as f:
    meta = json.load(f)

best_name = meta['best_model']
safe_name = best_name.lower().replace(' ', '_')
best_model = joblib.load(f'./models_saved/{safe_name}.pkl')

linear_models = ['Logistic Regression', 'SVM']
X_te = X_test_scaled if best_name in linear_models else X_test

threshold = float(meta['best_threshold'])

print(f"  Best Model: {best_name}")
print(f"  Threshold:  {threshold:.4f}")
print(f"  Test shape: {X_te.shape}")
print(f"  Fraud cases: {y_test.sum():,}")

FinGuard - Model Interpretability (SHAP)

[1] Loading data and best model...
  Best Model: Random Forest
  Threshold:  0.8775
  Test shape: (954393, 25)
  Fraud cases: 4,008


In [3]:
#  ============================================================
# 2. STRATIFIED SAMPLE FOR SHAP
# ============================================================
print("\n[2] Creating stratified sample for SHAP...")

NONFRAUD_SAMPLE = 5000

fraud_idx = y_test[y_test == 1].index
nonfraud_idx = y_test[y_test == 0].sample(
    NONFRAUD_SAMPLE, random_state=RANDOM_STATE
).index
sample_idx = fraud_idx.union(nonfraud_idx)

X_sample = X_te.loc[sample_idx].reset_index(drop=True)
y_sample = y_test.loc[sample_idx].reset_index(drop=True)

print(f"  SHAP sample: {X_sample.shape}")
print(f"    Fraud:     {y_sample.sum():,}")
print(f"    Non-Fraud: {(y_sample == 0).sum():,}")


[2] Creating stratified sample for SHAP...
  SHAP sample: (9008, 25)
    Fraud:     4,008
    Non-Fraud: 5,000


In [8]:
# ============================================================
# 3. UNWRAP MODEL (SAFELY)
# ============================================================
print("\n[3] Preparing model for SHAP...")

# --- Strategy: try unwrapping only for known wrappers, verify fitted ---
def is_fitted(model):
    """Check if sklearn model has been fitted."""
    from sklearn.utils.validation import check_is_fitted
    try:
        check_is_fitted(model)
        return True
    except Exception:
        return False


def unwrap_model(model, name):
    """
    Unwrap CalibratedClassifierCV to get the underlying estimator.
    Only unwrap if the inner estimator is fitted.
    """
    # Case 1: CalibratedClassifierCV -> get the first calibrated estimator
    if hasattr(model, 'calibrated_classifiers_') and \
       len(model.calibrated_classifiers_) > 0:
        inner = model.calibrated_classifiers_[0].estimator
        if is_fitted(inner):
            print(f"  Unwrapped CalibratedClassifierCV -> "
                  f"{type(inner).__name__}")
            return inner

    # Case 2: Pipeline -> get final step
    if hasattr(model, 'steps') and len(model.steps) > 0:
        inner = model.steps[-1][1]
        if is_fitted(inner):
            print(f"  Unwrapped Pipeline -> {type(inner).__name__}")
            return inner

    return model


model_obj = unwrap_model(best_model, best_name)

# Verify it's fitted
if not is_fitted(model_obj):
    raise RuntimeError(
        f"Model {type(model_obj).__name__} is not fitted! "
        f"Cannot explain an unfitted model."
    )

print(f"  Model ready: {type(model_obj).__name__}")



[3] Preparing model for SHAP...
  Model ready: RandomForestClassifier


In [9]:
# ============================================================
# 4. BUILD SHAP EXPLAINER
# ============================================================
print("\n[4] Building SHAP explainer...")

# Tree models that SHAP can handle natively
TREE_MODELS = [
    'RandomForestClassifier',
    'DecisionTreeClassifier',
    'GradientBoostingClassifier',
    'XGBClassifier',
    'LGBMClassifier',
    'XGBRegressor',
    'LGBMRegressor',
]

model_class_name = type(model_obj).__name__
print(f"  Detected class: {model_class_name}")

explainer = None
shap_values = None
explainer_type = None

# --- Try TreeExplainer first ---
if model_class_name in TREE_MODELS:
    try:
        print("  Trying TreeExplainer...")
        explainer = shap.TreeExplainer(model_obj)
        shap_values = explainer.shap_values(X_sample)
        explainer_type = "Tree"
        print("  ✓ TreeExplainer succeeded")
    except Exception as e:
        print(f"  ✗ TreeExplainer failed: {e}")
        explainer = None

# --- Fall back to LinearExplainer for linear models ---
if explainer is None and model_class_name in [
    'LogisticRegression', 'LinearSVC', 'SGDClassifier', 'RidgeClassifier'
]:
    try:
        print("  Trying LinearExplainer...")
        explainer = shap.LinearExplainer(model_obj, X_sample)
        shap_values = explainer.shap_values(X_sample)
        explainer_type = "Linear"
        print("  ✓ LinearExplainer succeeded")
    except Exception as e:
        print(f"  ✗ LinearExplainer failed: {e}")
        explainer = None

# --- Fall back to KernelExplainer (slow but universal) ---
if explainer is None:
    print("  Trying KernelExplainer (this may take a few minutes)...")

    # KernelExplainer needs a background sample
    bg_size = min(50, len(X_sample))
    bg = shap.sample(X_sample, bg_size, random_state=RANDOM_STATE)

    # Use the OUTER model (predict_proba works on wrappers)
    predict_fn = lambda x: best_model.predict_proba(x)[:, 1]

    explainer = shap.KernelExplainer(predict_fn, bg)
    explainer_type = "Kernel"

    # Use smaller subset for Kernel (it's O(n * 2^features))
    n_explain = min(200, len(X_sample))
    X_explain = X_sample.iloc[:n_explain].reset_index(drop=True)
    print(f"  KernelExplainer will explain {n_explain} samples "
          f"(features={X_explain.shape[1]})")

    shap_values = explainer.shap_values(X_explain, nsamples=100)

    # Align sample with explained subset
    X_sample = X_explain
    y_sample = y_sample.iloc[:n_explain].reset_index(drop=True)

    print("  ✓ KernelExplainer succeeded")

print(f"  Explainer used: {explainer_type}")


[4] Building SHAP explainer...
  Detected class: RandomForestClassifier
  Trying TreeExplainer...
  ✓ TreeExplainer succeeded
  Explainer used: Tree


In [10]:
# ============================================================
# 5. NORMALIZE SHAP VALUES
# ============================================================
print("\n[5] Normalizing SHAP values...")

# Different SHAP versions return different shapes:
#   - Old: list [class0_vals, class1_vals]
#   - New: ndarray (n, features, n_classes) or (n, features)
#   - Kernel: list or ndarray

if isinstance(shap_values, list):
    shap_values_fraud = shap_values[1]
elif isinstance(shap_values, np.ndarray):
    if shap_values.ndim == 3:
        shap_values_fraud = shap_values[:, :, 1]
    else:
        shap_values_fraud = shap_values
else:
    shap_values_fraud = np.asarray(shap_values)

print(f"  SHAP values shape: {shap_values_fraud.shape}")

# Ensure 2D
if shap_values_fraud.ndim != 2:
    raise RuntimeError(
        f"Unexpected SHAP shape: {shap_values_fraud.shape}"
    )

# Expected values (base rate)
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)) and \
   not np.isscalar(expected_value):
    expected_value = expected_value[1]
print(f"  Base value: {expected_value:.4f}")



[5] Normalizing SHAP values...
  SHAP values shape: (9008, 25)
  Base value: 0.5000


In [11]:
# ============================================================
# 6. GLOBAL FEATURE IMPORTANCE (BAR)
# ============================================================
print("\n[6] Global feature importance bar plot...")

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_fraud, X_sample,
    plot_type="bar", show=False, max_display=20
)
plt.title(f"SHAP Feature Importance - {best_name}",
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./reports/figures/fig9_shap_bar.png',
            dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ fig9_shap_bar.png")


# ============================================================
# 7. BEESWARM SUMMARY PLOT
# ============================================================
print("\n[7] SHAP beeswarm plot...")

plt.figure(figsize=(10, 10))
shap.summary_plot(
    shap_values_fraud, X_sample,
    show=False, max_display=20
)
plt.title(f"SHAP Summary - {best_name}",
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./reports/figures/fig10_shap_beeswarm.png',
            dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ fig10_shap_beeswarm.png")



[6] Global feature importance bar plot...
  ✓ fig9_shap_bar.png

[7] SHAP beeswarm plot...
  ✓ fig10_shap_beeswarm.png


In [12]:
# ============================================================
# 8. DEPENDENCE PLOTS (TOP 4)
# ============================================================
print("\n[8] SHAP dependence plots...")

mean_abs_shap = np.abs(shap_values_fraud).mean(axis=0)
top_features = list(X_sample.columns[
    np.argsort(mean_abs_shap)[-4:][::-1]
])
print(f"  Top 4: {top_features}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i in range(len(top_features)):
    feature = top_features[i]
    try:
        shap.dependence_plot(
            feature, shap_values_fraud, X_sample,
            ax=axes[i], show=False
        )
        axes[i].set_title(f"SHAP: {feature}", fontweight='bold')
    except Exception as e:
        axes[i].text(0.5, 0.5, f"Failed:\n{feature}",
                     ha='center', va='center', transform=axes[i].transAxes)

plt.tight_layout()
plt.savefig('./reports/figures/fig11_shap_dependence.png',
            dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ fig11_shap_dependence.png")


# ============================================================
# 9. WATERFALL PLOTS (Individual cases)
# ============================================================
print("\n[9] SHAP waterfall plots...")

# Get predictions on sample
y_proba = best_model.predict_proba(X_te.loc[sample_idx])[:, 1]
y_pred = (y_proba >= threshold).astype(int)

fraud_pos = np.where(y_sample.values == 1)[0]
tp_pos = [p for p in fraud_pos if y_pred[p] == 1]
fn_pos = [p for p in fraud_pos if y_pred[p] == 0]


def make_waterfall(idx, title, path):
    """Helper to create waterfall plot."""
    try:
        exp = shap.Explanation(
            values=shap_values_fraud[idx],
            base_values=expected_value,
            data=X_sample.iloc[idx].values,
            feature_names=X_sample.columns.tolist(),
        )
        plt.figure(figsize=(10, 6))
        shap.plots.waterfall(exp, max_display=15, show=False)
        plt.title(title, fontweight='bold')
        plt.tight_layout()
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()
        return True
    except Exception as e:
        print(f"    Waterfall failed: {e}")
        return False


if tp_pos:
    idx = tp_pos[0]
    ok = make_waterfall(
        idx,
        f"SHAP Waterfall - Caught Fraud (P={y_proba[idx]:.3f})",
        './reports/figures/fig12_shap_waterfall_tp.png'
    )
    if ok:
        print("  ✓ fig12_shap_waterfall_tp.png")

if fn_pos:
    idx = fn_pos[0]
    ok = make_waterfall(
        idx,
        f"SHAP Waterfall - Missed Fraud (P={y_proba[idx]:.3f})",
        './reports/figures/fig13_shap_waterfall_fn.png'
    )
    if ok:
        print("  ✓ fig13_shap_waterfall_fn.png")


[8] SHAP dependence plots...
  Top 4: ['amount_count', 'step_sin', 'is_payment', 'is_duplicate_amount']
  ✓ fig11_shap_dependence.png

[9] SHAP waterfall plots...
  ✓ fig12_shap_waterfall_tp.png
  ✓ fig13_shap_waterfall_fn.png


In [13]:
# ============================================================
# 10. SAVE FEATURE IMPORTANCE
# ============================================================
print("\n[10] Saving feature importance...")

importance_df = pd.DataFrame({
    'feature': X_sample.columns,
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
importance_df['rank'] = importance_df.index + 1

importance_df.to_csv(
    './reports/tables/shap_feature_importance.csv', index=False
)

top_10 = importance_df.head(10).to_dict(orient='records')
with open('./models_saved/shap_top_features.json', 'w') as f:
    json.dump(top_10, f, indent=4)

print("\n  Top 10 Features by SHAP:")
for row in top_10:
    print(f"    {row['rank']:2d}. {row['feature']:30s} "
          f"| SHAP={row['mean_abs_shap']:.4f}")

print("\n✅ Complete!")


# ============================================================
# 11. SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("INTERPRETATION SUMMARY")
print("=" * 60)
print(f"\nModel:   {best_name}")
print(f"Explainer: {explainer_type}")
print(f"Base value (log-odds): {expected_value:.4f}")
print("\nTop 5 drivers of fraud:")
for row in top_10[:5]:
    print(f"  • {row['feature']:30s} (SHAP={row['mean_abs_shap']:.4f})")
print("\nPositive SHAP → pushes toward FRAUD")
print("Negative SHAP → pushes toward NON-FRAUD")



[10] Saving feature importance...

  Top 10 Features by SHAP:
     1. amount_count                   | SHAP=0.1013
     2. step_sin                       | SHAP=0.0813
     3. is_payment                     | SHAP=0.0770
     4. is_duplicate_amount            | SHAP=0.0444
     5. is_cash_in                     | SHAP=0.0357
     6. amount                         | SHAP=0.0340
     7. type_encoded                   | SHAP=0.0315
     8. type_frequency                 | SHAP=0.0307
     9. log_amount                     | SHAP=0.0297
    10. hour_sin                       | SHAP=0.0290

✅ Complete!

INTERPRETATION SUMMARY

Model:   Random Forest
Explainer: Tree
Base value (log-odds): 0.5000

Top 5 drivers of fraud:
  • amount_count                   (SHAP=0.1013)
  • step_sin                       (SHAP=0.0813)
  • is_payment                     (SHAP=0.0770)
  • is_duplicate_amount            (SHAP=0.0444)
  • is_cash_in                     (SHAP=0.0357)

Positive SHAP → pushes toward